# Experimentos E1–E6 — Pedro, 23/10/2026

Resultados da frota definitiva e trânsito: três instâncias, cinco seeds,
300 gerações, 300 observações e 240 execuções únicas. A mesma execução de
referência atende mais de um experimento. Este notebook lê os resultados
versionados; não inicia uma rodada longa de otimização.

Reprodução: `python -m medroute.experiments.study --workers 4`.
Reanálise: `python -m medroute.experiments.study --render-only`.
Método e limitações: [seção do relatório](../reports/secoes/pedro-experimentos-s3.md).


In [1]:
import hashlib
import json
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RESULTS = ROOT / "reports/experimentos-s3"
metrics = pd.read_csv(RESULTS / "metrics.csv")
summary = pd.read_csv(RESULTS / "summary.csv")
solutions = json.loads((RESULTS / "solutions.json").read_text())
assert len(metrics) == 300 and len(solutions) == 240
assert set(summary["n"]) == {5}
print(f"{len(metrics)} observações; {len(solutions)} execuções únicas; {len(summary)} grupos.")
print(metrics.groupby(["experiment", "instance_nome"]).size().unstack().to_string())


300 observações; 240 execuções únicas; 60 grupos.
instance_nome  sp_15  sp_40  sp_80
experiment                        
E1                15     15     15
E2                15     15     15
E3                10     10     10
E4                15     15     15
E5                15     15     15
E6                30     30     30


## E6 — GA versus baselines

Fitness em R$ generalizados; menor é melhor. Desvio é amostral. V/E/D usa
as mesmas seeds contra NN+2-opt. Baselines determinísticos repetem o resultado;
cinco seeds não representam cinco evidências independentes desses métodos.


In [2]:
e6 = summary.loc[summary.experiment.eq("E6"), [
    "instance_nome", "configuration", "fitness_comum_media", "fitness_comum_dp",
    "fitness_comum_min", "melhoria_media_pct", "vitoria", "empate", "derrota",
    "execucoes_com_violacao_pct", "execucoes_com_violacao_hard_pct",
]]
print(e6.round(2).to_string(index=False))


instance_nome            configuration  fitness_comum_media  fitness_comum_dp  fitness_comum_min  melhoria_media_pct  vitoria  empate  derrota  execucoes_com_violacao_pct  execucoes_com_violacao_hard_pct
        sp_15                  hibrido               456.39              2.54             454.58                1.15        5       0        0                         0.0                              0.0
        sp_15         nearest_neighbor               474.44              0.00             474.44               -2.76        0       0        5                         0.0                              0.0
        sp_15 nearest_neighbor_two_opt               461.72              0.00             461.72                0.00        0       5        0                         0.0                              0.0
        sp_15                     puro               474.74             19.13             455.54               -2.82        2       0        3                         0.0              

![E6: fitness e convergência](../reports/figuras/s3/e6.png)

E6 usa escala logarítmica para manter visíveis as penalidades do aleatório.
As faixas das curvas indicam média ± desvio amostral entre as cinco seeds.


## E5 — separar custo operacional, prioridade e objetivo

Os perfis multiplicam os preços de prioridade por 0, 1 ou 3 e mantêm as
penalidades. Não se comparam fitness brutos de objetivos diferentes.
`fitness_comum` reavalia todas as rotas com os pesos oficiais.


In [3]:
e5 = summary.loc[summary.experiment.eq("E5"), [
    "instance_nome", "configuration", "fitness_media", "fitness_comum_media",
    "custo_operacional_media", "prioridade_comum_media", "chegada_critica_media_min_media",
    "criticas_atrasadas_media", "deadline_atraso_min_media",
]]
print(e5.round(2).to_string(index=False))


instance_nome    configuration  fitness_media  fitness_comum_media  custo_operacional_media  prioridade_comum_media  chegada_critica_media_min_media  criticas_atrasadas_media  deadline_atraso_min_media
        sp_15       custo_puro         273.82               574.90                   273.82                  301.08                            98.16                       0.0                       0.00
        sp_15      equilibrado         456.39               456.39                   306.10                  150.29                            51.00                       0.0                       0.00
        sp_15 prioridade_forte         714.49               497.54                   389.07                  108.47                            30.98                       0.0                       0.00
        sp_40       custo_puro         594.14              1261.41                   594.14                  667.26                            86.00                       0.0                  

![E5: componentes comparáveis](../reports/figuras/s3/e5_componentes.png)

## E1–E4 e rotas

As tabelas completas de média ± desvio, melhor solução, tempo e violações estão
em [resumo.md](../reports/experimentos-s3/resumo.md). Tempos foram medidos sob
concorrência; esta rodada não é um benchmark isolado de hardware.

![E1: população](../reports/figuras/s3/e1.png)
![E2: taxa](../reports/figuras/s3/e2.png)
![E3: crossover](../reports/figuras/s3/e3.png)
![E4: operador](../reports/figuras/s3/e4.png)
![Melhor híbrido em sp_80](../reports/figuras/s3/rotas_sp_80.png)

As rotas são segmentos entre coordenadas, sem trajetos viários. Distâncias usam
haversine × 1,30; cargas, destinos domiciliares e custos sem fonte são premissas
de simulação. As conclusões são descritivas destas instâncias e destas seeds.


In [4]:
validation = json.loads((RESULTS / "validacao.json").read_text())
for relative, expected in validation["sha256_artifacts"].items():
    actual = hashlib.sha256((ROOT / "reports" / relative).read_bytes()).hexdigest()
    assert actual == expected, relative
print(f"{validation['unique_solutions_validated']} soluções revalidadas pelo executor.")
print(f"{len(validation['sha256_artifacts'])} hashes de artefatos conferidos neste notebook.")


240 soluções revalidadas pelo executor.
41 hashes de artefatos conferidos neste notebook.
